In [ ]:
from pathlib import Path
import torch

print("===== ULTRAFEEDBACK RUNNER CHECK =====")

# Find the saved Version 4 repository
manifests = list(
    Path("/kaggle/input").rglob(
        "PA2_RESUME_STATE.json"
    )
)

print(
    "Saved manifests found:",
    len(manifests)
)

for p in manifests:
    print(
        "Manifest:",
        p
    )
    print(
        "Saved repo:",
        p.parent
    )


print(
    "\nCUDA available:",
    torch.cuda.is_available()
)

print(
    "GPU count:",
    torch.cuda.device_count()
)

for i in range(
    torch.cuda.device_count()
):
    print(
        f"GPU {i}:",
        torch.cuda.get_device_name(i)
    )

In [ ]:
# ============================================================
# ULTRAFEEDBACK DPO RUNNER — SETUP
# ============================================================

from pathlib import Path
import os
import sys
import gc
import json
import random
import hashlib
import shutil
import importlib

import numpy as np
import torch
import transformers

from torch.utils.data import (
    Dataset,
    DataLoader,
)

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)


# ============================================================
# ENVIRONMENT
# ============================================================

print("===== ENVIRONMENT =====")

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)

print(
    "CUDA available:",
    torch.cuda.is_available(),
)

print(
    "GPU count:",
    torch.cuda.device_count(),
)

assert torch.cuda.is_available()
assert torch.cuda.device_count() >= 2

for i in range(
    torch.cuda.device_count()
):
    print(
        f"GPU {i}:",
        torch.cuda.get_device_name(i),
    )


POLICY_DEVICE = torch.device(
    "cuda:0"
)

REFERENCE_DEVICE = torch.device(
    "cuda:1"
)


# ============================================================
# FIND SAVED VERSION 4
# ============================================================

manifest_candidates = [
    p
    for p in Path(
        "/kaggle/input"
    ).rglob(
        "PA2_RESUME_STATE.json"
    )
    if "adv-gen-ai-pa2" in str(p)
]


print(
    "\nManifest candidates:",
    len(manifest_candidates),
)

for p in manifest_candidates:
    print(p)


assert len(manifest_candidates) == 1, (
    "Expected exactly one saved PA2 manifest."
)


SAVED_MANIFEST = (
    manifest_candidates[0]
)

SAVED_REPO = (
    SAVED_MANIFEST.parent
)


WORKING_REPO = Path(
    "/kaggle/working/"
    "small-model-alignment-starter"
)


print(
    "\nSaved repo:",
    SAVED_REPO,
)

print(
    "Working repo:",
    WORKING_REPO,
)


# ============================================================
# RESTORE VERSION 4
# ============================================================

if WORKING_REPO.exists():
    shutil.rmtree(
        WORKING_REPO
    )


print(
    "\nCopying Version 4 into writable storage..."
)

shutil.copytree(
    SAVED_REPO,
    WORKING_REPO,
)

print(
    "✓ Version 4 restored"
)


os.chdir(
    WORKING_REPO
)

sys.path.insert(
    0,
    str(WORKING_REPO),
)


# ============================================================
# PATCH OLD PROGRESS-BAR LABEL ONLY
# ============================================================

DPO_FILE = (
    WORKING_REPO
    / "dpo_training.py"
)

dpo_text = DPO_FILE.read_text(
    encoding="utf-8"
)

old_label = (
    'desc="HH DPO updates",'
)

new_label = (
    'desc=f"{config[\'run_name\']} updates",'
)


if old_label in dpo_text:

    dpo_text = dpo_text.replace(
        old_label,
        new_label,
    )

    DPO_FILE.write_text(
        dpo_text,
        encoding="utf-8",
    )

    print(
        "✓ Progress label patched"
    )

else:

    print(
        "✓ Progress label already generic"
    )


# ============================================================
# IMPORT OUR TESTED IMPLEMENTATION
# ============================================================

import data_utils
import dpo_training

data_utils = importlib.reload(
    data_utils
)

dpo_training = importlib.reload(
    dpo_training
)


from data_utils import (
    format_instruction_prompt,
)

from dpo_training import (
    train_dpo,
)


# ============================================================
# PATHS
# ============================================================

SFT_BEST_DIR = (
    WORKING_REPO
    / "outputs"
    / "sft_qwen2.5_0.5b_ultrachat_seed42"
    / "best_checkpoint"
)


UF_TRAIN_PATH = (
    WORKING_REPO
    / "data"
    / "ultrafeedback_fixed"
    / "ultrafeedback_train_15000_seed42.jsonl"
)


UF_VAL_PATH = (
    WORKING_REPO
    / "data"
    / "ultrafeedback_fixed"
    / "ultrafeedback_val_1000_seed42.jsonl"
)


UF_OUTPUT_DIR = (
    WORKING_REPO
    / "outputs"
    / "ultrafeedback_dpo_seed42"
)


UF_CONFIG_PATH = (
    UF_OUTPUT_DIR
    / "config.json"
)


assert SFT_BEST_DIR.exists()
assert UF_TRAIN_PATH.exists()
assert UF_VAL_PATH.exists()
assert UF_CONFIG_PATH.exists()


# ============================================================
# HASH VERIFICATION
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


uf_config = json.loads(
    UF_CONFIG_PATH.read_text(
        encoding="utf-8"
    )
)


assert (
    sha256_file(
        UF_TRAIN_PATH
    )
    ==
    uf_config[
        "train_data_sha256"
    ]
)


assert (
    sha256_file(
        UF_VAL_PATH
    )
    ==
    uf_config[
        "validation_data_sha256"
    ]
)


print(
    "✓ UltraFeedback data hashes verified"
)


# ============================================================
# RECORD ACTUAL PREFLIGHT + ENVIRONMENT
# ============================================================

uf_config[
    "ultrafeedback_amp_preflight"
] = {
    "pairs": 32,

    "mean_dpo_loss": (
        0.6931471824645996
    ),

    "gradient_tensors": 290,

    "nonfinite_gradient_tensors": 0,

    "all_gradients_finite": True,

    "max_abs_unscaled_gradient": (
        3.4931087493896484
    ),

    "gradient_norm_before_clip": (
        61.619895935058594
    ),

    "amp_init_scale": 512.0,

    "policy_peak_gpu_gb": (
        7.5375189781188965
    ),

    "reference_peak_gpu_gb": (
        3.2146735191345215
    ),

    "optimizer_step_performed": False,
}


uf_config[
    "environment"
] = {
    "torch": torch.__version__,
    "transformers": (
        transformers.__version__
    ),
}


# Ensure paths refer to THIS restored run.
uf_config[
    "initial_policy"
] = str(
    SFT_BEST_DIR
)

uf_config[
    "reference_model"
] = str(
    SFT_BEST_DIR
)


UF_CONFIG_PATH.write_text(
    json.dumps(
        uf_config,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# REPRODUCIBILITY
# ============================================================

SEED = int(
    uf_config["seed"]
)

random.seed(
    SEED
)

np.random.seed(
    SEED
)

torch.manual_seed(
    SEED
)

torch.cuda.manual_seed_all(
    SEED
)


# ============================================================
# DATASET
# ============================================================

class PreferenceDataset(Dataset):

    def __init__(
        self,
        path,
    ):

        self.rows = []

        with open(
            path,
            "r",
            encoding="utf-8",
        ) as f:

            for line in f:

                self.rows.append(
                    json.loads(
                        line
                    )
                )


    def __len__(
        self
    ):

        return len(
            self.rows
        )


    def __getitem__(
        self,
        index,
    ):

        row = self.rows[
            index
        ]

        return {
            "prompts": (
                format_instruction_prompt(
                    row["instruction"]
                )
            ),

            "chosen": (
                row["chosen"]
            ),

            "rejected": (
                row["rejected"]
            ),
        }


train_dataset = PreferenceDataset(
    UF_TRAIN_PATH
)

val_dataset = PreferenceDataset(
    UF_VAL_PATH
)


assert len(
    train_dataset
) == 15000

assert len(
    val_dataset
) == 1000


print(
    "\nTrain pairs:",
    len(train_dataset),
)

print(
    "Validation pairs:",
    len(val_dataset),
)


# ============================================================
# DATALOADERS
# ============================================================

train_generator = (
    torch.Generator()
)

train_generator.manual_seed(
    SEED
)


train_loader = DataLoader(
    train_dataset,

    batch_size=1,

    shuffle=True,

    generator=train_generator,

    num_workers=0,

    drop_last=False,
)


val_loader = DataLoader(
    val_dataset,

    batch_size=1,

    shuffle=False,

    num_workers=0,

    drop_last=False,
)


assert len(
    train_loader
) == 15000

assert len(
    val_loader
) == 1000


# Verify the structure expected by train_dpo().
sample_batch = next(
    iter(
        val_loader
    )
)


print(
    "\nBatch keys:",
    sample_batch.keys(),
)

print(
    "Prompt batch size:",
    len(
        sample_batch[
            "prompts"
        ]
    ),
)

print(
    "Chosen batch size:",
    len(
        sample_batch[
            "chosen"
        ]
    ),
)

print(
    "Rejected batch size:",
    len(
        sample_batch[
            "rejected"
        ]
    ),
)


assert (
    len(
        sample_batch[
            "prompts"
        ]
    )
    == 1
)


# ============================================================
# TOKENIZER
# ============================================================

tokenizer = (
    AutoTokenizer.from_pretrained(
        SFT_BEST_DIR,
        use_fast=True,
    )
)


# ============================================================
# ENSURE THIS IS A FRESH UF TRAINING RUN
# ============================================================

for path in [
    UF_OUTPUT_DIR
    / "metrics.jsonl",

    UF_OUTPUT_DIR
    / "summary.json",

    UF_OUTPUT_DIR
    / "best_checkpoint",

    UF_OUTPUT_DIR
    / "final_checkpoint",
]:

    if path.exists():

        raise RuntimeError(
            "UltraFeedback training output "
            "already exists:\n"
            f"{path}"
        )


print(
    "\n✓ Fresh UltraFeedback output directory"
)


# ============================================================
# LOAD POLICY
# ============================================================

gc.collect()
torch.cuda.empty_cache()


print(
    "\n===== LOADING POLICY FROM SFT BEST ====="
)

policy_model = (
    AutoModelForCausalLM
    .from_pretrained(
        SFT_BEST_DIR,
        dtype=torch.float32,
    )
)

policy_model.to(
    POLICY_DEVICE
)

policy_model.train()


# ============================================================
# LOAD FROZEN REFERENCE
# ============================================================

print(
    "\n===== LOADING REFERENCE FROM SAME SFT BEST ====="
)

reference_model = (
    AutoModelForCausalLM
    .from_pretrained(
        SFT_BEST_DIR,
        dtype=torch.float32,
    )
)

reference_model.to(
    REFERENCE_DEVICE
)

reference_model.eval()

reference_model.requires_grad_(
    False
)


assert any(
    p.requires_grad
    for p in policy_model.parameters()
)

assert not any(
    p.requires_grad
    for p in reference_model.parameters()
)


print(
    "\n✓ Policy = original SFT best"
)

print(
    "✓ Reference = original SFT best"
)

print(
    "✓ Reference frozen"
)

print(
    "✓ HH-DPO checkpoint is not used"
)

print(
    "✓ Expected optimizer updates:",
    uf_config[
        "total_update_steps"
    ],
)

print(
    "\n===== SETUP COMPLETE ====="
)

In [ ]:
# ============================================================
# FULL ULTRAFEEDBACK DPO TRAINING
# ============================================================

import torch


print(
    "=========================================="
)

print(
    "STARTING FULL ULTRAFEEDBACK DPO TRAINING"
)

print(
    "=========================================="
)

print(
    "Train preference pairs:",
    len(train_loader),
)

print(
    "Validation preference pairs:",
    len(val_loader),
)

print(
    "Gradient accumulation:",
    uf_config[
        "gradient_accumulation_steps"
    ],
)

print(
    "Expected updates:",
    uf_config[
        "total_update_steps"
    ],
)

print(
    "Learning rate:",
    uf_config[
        "learning_rate"
    ],
)

print(
    "Beta:",
    uf_config[
        "beta"
    ],
)

print(
    "AMP initial scale:",
    uf_config[
        "amp_init_scale"
    ],
)


torch.cuda.reset_peak_memory_stats(
    POLICY_DEVICE
)

torch.cuda.reset_peak_memory_stats(
    REFERENCE_DEVICE
)


uf_training_result = train_dpo(
    policy_model=policy_model,

    reference_model=reference_model,

    tokenizer=tokenizer,

    train_loader=train_loader,

    val_loader=val_loader,

    config=uf_config,

    output_dir=UF_OUTPUT_DIR,

    policy_device=POLICY_DEVICE,

    reference_device=REFERENCE_DEVICE,
)


policy_peak_gb = (
    torch.cuda.max_memory_allocated(
        POLICY_DEVICE
    )
    / (1024 ** 3)
)

reference_peak_gb = (
    torch.cuda.max_memory_allocated(
        REFERENCE_DEVICE
    )
    / (1024 ** 3)
)


print(
    "\n===== GPU PEAK MEMORY ====="
)

print(
    "Policy GPU peak GB:",
    policy_peak_gb,
)

print(
    "Reference GPU peak GB:",
    reference_peak_gb,
)


print(
    "\n✓ FULL ULTRAFEEDBACK TRAINING RETURNED SUCCESSFULLY"
)

In [ ]:
# ============================================================
# POST-TRAINING VERIFICATION + RESUME STATE UPDATE
# ============================================================

from pathlib import Path
import json
import hashlib


ROOT = Path(
    "/kaggle/working/small-model-alignment-starter"
)

UF_DIR = (
    ROOT
    / "outputs"
    / "ultrafeedback_dpo_seed42"
)

MANIFEST_PATH = (
    ROOT
    / "PA2_RESUME_STATE.json"
)


def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


# ============================================================
# REQUIRED OUTPUTS
# ============================================================

required_outputs = [
    (
        UF_DIR
        / "config.json"
    ),

    (
        UF_DIR
        / "metrics.jsonl"
    ),

    (
        UF_DIR
        / "summary.json"
    ),

    (
        UF_DIR
        / "best_checkpoint"
        / "model.safetensors"
    ),

    (
        UF_DIR
        / "final_checkpoint"
        / "model.safetensors"
    ),
]


print(
    "===== VERIFYING ULTRAFEEDBACK OUTPUTS ====="
)


for path in required_outputs:

    print(
        path.relative_to(ROOT),
        "->",
        path.exists(),
    )

    assert path.exists()


# ============================================================
# SUMMARY CHECK
# ============================================================

summary = json.loads(
    (
        UF_DIR
        / "summary.json"
    ).read_text(
        encoding="utf-8"
    )
)


print(
    "\n===== ULTRAFEEDBACK TRAINING SUMMARY ====="
)

print(
    json.dumps(
        summary,
        indent=2,
    )
)


assert summary[
    "completed"
] is True

assert summary[
    "optimizer_updates"
] == 469

assert summary[
    "expected_optimizer_updates"
] == 469


# ============================================================
# METRICS RECORD COUNTS
# ============================================================

metrics = []

with open(
    UF_DIR
    / "metrics.jsonl",
    "r",
    encoding="utf-8",
) as f:

    for line in f:

        metrics.append(
            json.loads(
                line
            )
        )


initial_records = [
    x for x in metrics
    if x["type"]
    == "initial_validation"
]

train_records = [
    x for x in metrics
    if x["type"]
    == "train_update"
]

validation_records = [
    x for x in metrics
    if x["type"]
    == "validation"
]


print(
    "\nMetrics records:",
    len(metrics),
)

print(
    "Initial validation:",
    len(initial_records),
)

print(
    "Train updates:",
    len(train_records),
)

print(
    "Validation records:",
    len(validation_records),
)


assert len(
    initial_records
) == 1

assert len(
    train_records
) == 469

assert len(
    validation_records
) == 5


# ============================================================
# MODEL HASHES
# ============================================================

best_model = (
    UF_DIR
    / "best_checkpoint"
    / "model.safetensors"
)

final_model = (
    UF_DIR
    / "final_checkpoint"
    / "model.safetensors"
)


best_sha = sha256_file(
    best_model
)

final_sha = sha256_file(
    final_model
)


print(
    "\nBest model SHA256:"
)

print(
    best_sha
)

print(
    "\nFinal model SHA256:"
)

print(
    final_sha
)


# ============================================================
# UPDATE RESUME MANIFEST
# ============================================================

state = json.loads(
    MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)


# Re-hash everything already tracked,
# because config/code may have changed.
for rel_path in list(
    state[
        "critical_files"
    ].keys()
):

    path = (
        ROOT
        / rel_path
    )

    if not path.exists():

        raise RuntimeError(
            "Tracked artifact missing: "
            f"{rel_path}"
        )


    state[
        "critical_files"
    ][
        rel_path
    ] = {
        "sha256": (
            sha256_file(
                path
            )
        )
    }


new_critical_files = [
    (
        "outputs/"
        "ultrafeedback_dpo_seed42/"
        "metrics.jsonl"
    ),

    (
        "outputs/"
        "ultrafeedback_dpo_seed42/"
        "summary.json"
    ),

    (
        "outputs/"
        "ultrafeedback_dpo_seed42/"
        "best_checkpoint/"
        "model.safetensors"
    ),

    (
        "outputs/"
        "ultrafeedback_dpo_seed42/"
        "final_checkpoint/"
        "model.safetensors"
    ),
]


for rel_path in (
    new_critical_files
):

    path = (
        ROOT
        / rel_path
    )

    state[
        "critical_files"
    ][
        rel_path
    ] = {
        "sha256": (
            sha256_file(
                path
            )
        )
    }


state[
    "current_state"
] = (
    "SFT, HH-RLHF from-scratch DPO, and "
    "UltraFeedback from-scratch DPO are complete. "
    "UltraFeedback completed all 469 optimizer updates "
    "and produced verified best/final checkpoints, "
    "metrics, and summary."
)


state[
    "next_stage"
] = (
    "Evaluate UltraFeedback DPO on the fixed "
    "AlpacaEval-300 set using exactly the same frozen "
    "decoding configuration as the existing base/SFT "
    "evaluation, then continue with the remaining "
    "assignment experiments."
)


MANIFEST_PATH.write_text(
    json.dumps(
        state,
        indent=2,
    ),
    encoding="utf-8",
)


print(
    "\nCritical files now:",
    len(
        state[
            "critical_files"
        ]
    ),
)

print(
    "Updated manifest SHA256:"
)

print(
    sha256_file(
        MANIFEST_PATH
    )
)


# ============================================================
# COMPLETION MARKER
# ============================================================

completion_marker = (
    ROOT
    / "ULTRAFEEDBACK_DPO_COMPLETE.txt"
)


completion_marker.write_text(
    (
        "UltraFeedback DPO completed successfully.\n"
        f"Best update: {summary['best_update']}\n"
        f"Best reward accuracy: "
        f"{summary['best_reward_accuracy']}\n"
        f"Best reward margin: "
        f"{summary['best_reward_margin']}\n"
        f"Best SHA256: {best_sha}\n"
        f"Final SHA256: {final_sha}\n"
    ),
    encoding="utf-8",
)


print(
    "\n=========================================="
)

print(
    "✓ ULTRAFEEDBACK DPO RUN FULLY COMPLETE"
)

print(
    "=========================================="
)

print(
    "Best update:",
    summary[
        "best_update"
    ],
)

print(
    "Best reward accuracy:",
    summary[
        "best_reward_accuracy"
    ],
)

print(
    "Final validation:",
    summary[
        "final_validation"
    ],
)

print(
    "\nSafe completion marker:",
    completion_marker,
)

In [ ]:
from pathlib import Path

print("===== FINDING COMPLETED ULTRAFEEDBACK RUN =====")

manifests = list(
    Path("/kaggle/input").rglob(
        "PA2_RESUME_STATE.json"
    )
)

markers = list(
    Path("/kaggle/input").rglob(
        "ULTRAFEEDBACK_DPO_COMPLETE.txt"
    )
)

print(
    "Manifests found:",
    len(manifests)
)

for p in manifests:
    print(
        "MANIFEST:",
        p
    )

print(
    "\nCompletion markers found:",
    len(markers)
)

for p in markers:
    print(
        "MARKER:",
        p
    )

In [ ]:
from pathlib import Path
import shutil
import json
import hashlib


# ============================================================
# COMPLETED ULTRAFEEDBACK RUN
# ============================================================

SAVED_REPO = Path(
    "/kaggle/input/notebooks/"
    "muhammadhashirsalman/"
    "notebook06adee0aec/"
    "small-model-alignment-starter"
)

WORKING_REPO = Path(
    "/kaggle/working/"
    "small-model-alignment-starter"
)


print("===== RESTORING COMPLETED ULTRAFEEDBACK RUN =====")

print(
    "Source:",
    SAVED_REPO
)

print(
    "Destination:",
    WORKING_REPO
)


assert (
    SAVED_REPO
    / "ULTRAFEEDBACK_DPO_COMPLETE.txt"
).exists()

assert (
    SAVED_REPO
    / "PA2_RESUME_STATE.json"
).exists()


# ============================================================
# SAFETY
# ============================================================

if WORKING_REPO.exists():
    raise RuntimeError(
        f"{WORKING_REPO} already exists.\n"
        "Do not overwrite it. Send me this message if it appears."
    )


# ============================================================
# COPY TO WRITABLE STORAGE
# ============================================================

print(
    "\nCopying completed repository..."
)

shutil.copytree(
    SAVED_REPO,
    WORKING_REPO,
)

print(
    "✓ Repository restored"
)


# ============================================================
# LOAD UPDATED MANIFEST
# ============================================================

MANIFEST = (
    WORKING_REPO
    / "PA2_RESUME_STATE.json"
)

state = json.loads(
    MANIFEST.read_text(
        encoding="utf-8"
    )
)


print(
    "\nCurrent state:"
)

print(
    state["current_state"]
)

print(
    "\nNext stage:"
)

print(
    state["next_stage"]
)


# ============================================================
# VERIFY ALL TRACKED FILE HASHES
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


print(
    "\n===== VERIFYING RESTORED ARTIFACTS ====="
)

failures = []


for relative_path, info in (
    state["critical_files"].items()
):

    path = (
        WORKING_REPO
        / relative_path
    )

    if not path.exists():

        print(
            "✗ MISSING:",
            relative_path
        )

        failures.append(
            relative_path
        )

        continue


    actual_sha = sha256_file(
        path
    )


    if (
        actual_sha
        != info["sha256"]
    ):

        print(
            "✗ HASH MISMATCH:",
            relative_path
        )

        failures.append(
            relative_path
        )

    else:

        print(
            "✓",
            relative_path
        )


print(
    "\n===== RESTORE RESULT ====="
)

print(
    "Critical files:",
    len(
        state["critical_files"]
    )
)

print(
    "Failures:",
    len(failures)
)


if failures:

    raise RuntimeError(
        "Restore verification failed."
    )


print(
    "\n✓ COMPLETED ULTRAFEEDBACK RUN FULLY RESTORED"
)


# ============================================================
# ULTRAFEEDBACK RESULT QUICK CHECK
# ============================================================

UF_SUMMARY = (
    WORKING_REPO
    / "outputs"
    / "ultrafeedback_dpo_seed42"
    / "summary.json"
)

uf_summary = json.loads(
    UF_SUMMARY.read_text(
        encoding="utf-8"
    )
)


print(
    "\n===== ULTRAFEEDBACK RESULT ====="
)

print(
    "Best update:",
    uf_summary["best_update"]
)

print(
    "Best reward accuracy:",
    uf_summary[
        "best_reward_accuracy"
    ]
)

print(
    "Best reward margin:",
    uf_summary[
        "best_reward_margin"
    ]
)

print(
    "Final validation:",
    uf_summary[
        "final_validation"
    ]
)

In [ ]:
import json
from pathlib import Path

ROOT = Path(
    "/kaggle/working/small-model-alignment-starter"
)

ALPACA_PATH = (
    ROOT
    / "eval_sets"
    / "alpaca_eval_300_seed42.json"
)

SFT_OUTPUT_PATH = (
    ROOT
    / "outputs"
    / "eval"
    / "sft_qwen2.5_0.5b"
    / "alpaca_eval_300_seed42_sft.json"
)

BASE_OUTPUT_PATH = (
    ROOT
    / "outputs"
    / "eval"
    / "base_qwen2.5_0.5b"
    / "alpaca_eval_300_seed42_base_nr4.json"
)


# ============================================================
# LOAD
# ============================================================

alpaca_rows = json.loads(
    ALPACA_PATH.read_text(
        encoding="utf-8"
    )
)

sft_rows = json.loads(
    SFT_OUTPUT_PATH.read_text(
        encoding="utf-8"
    )
)

base_rows = json.loads(
    BASE_OUTPUT_PATH.read_text(
        encoding="utf-8"
    )
)


print("===== COUNTS =====")
print("Fixed prompts:", len(alpaca_rows))
print("SFT outputs:", len(sft_rows))
print("Base outputs:", len(base_rows))


print("\n===== FIXED PROMPT ROW 0 =====")
print(
    json.dumps(
        alpaca_rows[0],
        indent=2,
        ensure_ascii=False,
    )
)


print("\n===== SFT OUTPUT ROW 0 =====")
print(
    json.dumps(
        sft_rows[0],
        indent=2,
        ensure_ascii=False,
    )
)


print("\n===== BASE OUTPUT ROW 0 =====")
print(
    json.dumps(
        base_rows[0],
        indent=2,
        ensure_ascii=False,
    )
)


print("\n===== OUTPUT KEYS =====")
print(
    "SFT:",
    sft_rows[0].keys()
)

print(
    "Base:",
    base_rows[0].keys()
)

In [ ]:
# ============================================================
# ULTRAFEEDBACK-DPO -> ALPACAEVAL-300
# ============================================================

from pathlib import Path
import json
import hashlib
import os
import shutil
import sys
import gc

import torch
from tqdm.auto import tqdm
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)


# ============================================================
# FIND COMPLETED ULTRAFEEDBACK REPOSITORY
# ============================================================

candidates = [
    p
    for p in Path("/kaggle/input").rglob(
        "ULTRAFEEDBACK_DPO_COMPLETE.txt"
    )
]

print(
    "Completion markers found:",
    len(candidates)
)

for p in candidates:
    print(p)


assert len(candidates) == 1


SAVED_REPO = (
    candidates[0].parent
)

WORKING_REPO = Path(
    "/kaggle/working/"
    "small-model-alignment-starter"
)


print(
    "\nSaved repo:",
    SAVED_REPO
)

print(
    "Working repo:",
    WORKING_REPO
)


# ============================================================
# RESTORE TO WRITABLE STORAGE
# ============================================================

if WORKING_REPO.exists():
    shutil.rmtree(
        WORKING_REPO
    )


print(
    "\nRestoring completed PA2 repository..."
)

shutil.copytree(
    SAVED_REPO,
    WORKING_REPO,
)

print(
    "✓ Repository restored"
)


os.chdir(
    WORKING_REPO
)

sys.path.insert(
    0,
    str(WORKING_REPO),
)


from data_utils import (
    format_instruction_prompt,
)


# ============================================================
# PATHS
# ============================================================

UF_BEST = (
    WORKING_REPO
    / "outputs"
    / "ultrafeedback_dpo_seed42"
    / "best_checkpoint"
)

EVAL_PATH = (
    WORKING_REPO
    / "eval_sets"
    / "alpaca_eval_300_seed42.json"
)

OUT_DIR = (
    WORKING_REPO
    / "outputs"
    / "eval"
    / "ultrafeedback_dpo_seed42"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

OUT_PATH = (
    OUT_DIR
    / "alpaca_eval_300_seed42_ultrafeedback_dpo.json"
)


assert UF_BEST.exists()
assert EVAL_PATH.exists()

if OUT_PATH.exists():
    raise RuntimeError(
        f"Output already exists: {OUT_PATH}"
    )


# ============================================================
# DEVICE
# ============================================================

assert torch.cuda.is_available()

DEVICE = torch.device(
    "cuda:0"
)

print(
    "\nGPU:",
    torch.cuda.get_device_name(0)
)


# ============================================================
# LOAD EVALUATION SET
# ============================================================

eval_rows = json.loads(
    EVAL_PATH.read_text(
        encoding="utf-8"
    )
)

assert len(eval_rows) == 300


print(
    "Evaluation prompts:",
    len(eval_rows)
)


# ============================================================
# LOAD TOKENIZER + ULTRAFEEDBACK-DPO BEST MODEL
# ============================================================

tokenizer = (
    AutoTokenizer.from_pretrained(
        UF_BEST,
        use_fast=True,
    )
)


print(
    "\nLoading UltraFeedback-DPO best checkpoint..."
)

model = (
    AutoModelForCausalLM.from_pretrained(
        UF_BEST,
        dtype=torch.float32,
    )
)

model.to(
    DEVICE
)

model.eval()


print(
    "✓ Model loaded"
)


# ============================================================
# FROZEN DECODING
# EXACTLY SAME AS BASE / SFT
# ============================================================

GENERATION_CONFIG = {
    "do_sample": False,
    "max_new_tokens": 256,
    "repetition_penalty": 1.0,
    "no_repeat_ngram_size": 4,
}


print(
    "\nGeneration config:"
)

print(
    json.dumps(
        GENERATION_CONFIG,
        indent=2,
    )
)


# ============================================================
# GENERATE
# ============================================================

outputs = []


with torch.inference_mode():

    for row in tqdm(
        eval_rows,
        desc="UltraFeedback-DPO AlpacaEval",
    ):

        instruction = (
            row["instruction"]
        )

        prompt = (
            format_instruction_prompt(
                instruction
            )
        )


        encoded = tokenizer(
            prompt,
            return_tensors="pt",
            add_special_tokens=False,
        )


        prompt_token_count = (
            encoded[
                "input_ids"
            ].shape[1]
        )


        encoded = {
            key: value.to(
                DEVICE
            )
            for key, value
            in encoded.items()
        }


        generated = model.generate(
            **encoded,

            do_sample=False,

            max_new_tokens=256,

            repetition_penalty=1.0,

            no_repeat_ngram_size=4,

            pad_token_id=(
                tokenizer.pad_token_id
            ),

            eos_token_id=(
                tokenizer.eos_token_id
            ),
        )


        new_tokens = generated[
            0,
            prompt_token_count:
        ]


        generated_token_count = int(
            new_tokens.shape[0]
        )


        output_text = tokenizer.decode(
            new_tokens,
            skip_special_tokens=True,
        ).strip()


        hit_max_new_tokens = (
            generated_token_count
            >= 256
        )


        outputs.append({
            "id": (
                row["id"]
            ),

            "instruction": (
                instruction
            ),

            "output": (
                output_text
            ),

            "generator": (
                "qwen2.5-0.5b-"
                "ultrafeedback-dpo"
            ),

            "dataset": (
                "alpaca_eval"
            ),

            "source_index": (
                row["source_index"]
            ),

            "subset_position": (
                row["subset_position"]
            ),

            "prompt_token_count": (
                prompt_token_count
            ),

            "generated_token_count": (
                generated_token_count
            ),

            "hit_max_new_tokens": (
                hit_max_new_tokens
            ),
        })


# ============================================================
# SAVE
# ============================================================

OUT_PATH.write_text(
    json.dumps(
        outputs,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


print(
    "\n✓ Saved:",
    OUT_PATH
)

print(
    "Rows:",
    len(outputs)
)

assert len(outputs) == 300

In [ ]:
# ============================================================
# VERIFY + SUMMARIZE ALPACAEVAL GENERATION
# ============================================================

from pathlib import Path
import json
import hashlib
import statistics


ROOT = Path(
    "/kaggle/working/"
    "small-model-alignment-starter"
)

OUT_PATH = (
    ROOT
    / "outputs"
    / "eval"
    / "ultrafeedback_dpo_seed42"
    / "alpaca_eval_300_seed42_ultrafeedback_dpo.json"
)

MANIFEST = (
    ROOT
    / "PA2_RESUME_STATE.json"
)


rows = json.loads(
    OUT_PATH.read_text(
        encoding="utf-8"
    )
)

assert len(rows) == 300


# ============================================================
# REPEATED 4-GRAM FRACTION
# ============================================================

def repeated_ngram_fraction(
    text,
    n=4,
):

    tokens = text.split()

    if len(tokens) < n:
        return 0.0

    ngrams = [
        tuple(
            tokens[i:i+n]
        )
        for i in range(
            len(tokens) - n + 1
        )
    ]

    if not ngrams:
        return 0.0

    unique = set(
        ngrams
    )

    repeated_count = (
        len(ngrams)
        - len(unique)
    )

    return (
        repeated_count
        / len(ngrams)
    )


generated_lengths = [
    row[
        "generated_token_count"
    ]
    for row in rows
]

capped = sum(
    bool(
        row[
            "hit_max_new_tokens"
        ]
    )
    for row in rows
)

repeat_fracs = [
    repeated_ngram_fraction(
        row["output"],
        n=4,
    )
    for row in rows
]


summary = {
    "examples": (
        len(rows)
    ),

    "hit_max_new_tokens": (
        capped
    ),

    "hit_max_new_tokens_fraction": (
        capped / len(rows)
    ),

    "average_generated_tokens": (
        sum(generated_lengths)
        / len(generated_lengths)
    ),

    "mean_repeated_4gram_fraction": (
        sum(repeat_fracs)
        / len(repeat_fracs)
    ),

    "repeated_4gram_fraction_over_20pct": (
        sum(
            x > 0.20
            for x in repeat_fracs
        )
    ),

    "repeated_4gram_fraction_over_40pct": (
        sum(
            x > 0.40
            for x in repeat_fracs
        )
    ),

    "decoding": {
        "do_sample": False,
        "max_new_tokens": 256,
        "repetition_penalty": 1.0,
        "no_repeat_ngram_size": 4,
    },
}


print(
    "===== ULTRAFEEDBACK-DPO ALPACAEVAL SUMMARY ====="
)

print(
    json.dumps(
        summary,
        indent=2,
    )
)


# ============================================================
# HASH
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


output_sha = sha256_file(
    OUT_PATH
)


print(
    "\nSHA256:"
)

print(
    output_sha
)


# ============================================================
# UPDATE RESUME MANIFEST
# ============================================================

state = json.loads(
    MANIFEST.read_text(
        encoding="utf-8"
    )
)


relative_output = (
    "outputs/eval/"
    "ultrafeedback_dpo_seed42/"
    "alpaca_eval_300_seed42_"
    "ultrafeedback_dpo.json"
)


state[
    "critical_files"
][
    relative_output
] = {
    "sha256": (
        output_sha
    )
}


state[
    "current_state"
] = (
    "SFT, HH-RLHF DPO, and UltraFeedback DPO "
    "training are complete. UltraFeedback-DPO "
    "generation on the fixed AlpacaEval-300 "
    "evaluation set is also complete using the "
    "same frozen decoding as the Base/SFT runs."
)


state[
    "next_stage"
] = (
    "Compare SFT vs UltraFeedback-DPO outputs "
    "on AlpacaEval-300 with the required pairwise "
    "judge, and continue remaining PA2 experiments."
)


MANIFEST.write_text(
    json.dumps(
        state,
        indent=2,
    ),
    encoding="utf-8",
)


print(
    "\nCritical files:",
    len(
        state[
            "critical_files"
        ]
    ),
)


# ============================================================
# COMPLETION MARKER
# ============================================================

marker = (
    ROOT
    / "ULTRAFEEDBACK_ALPACAEVAL_COMPLETE.txt"
)


marker.write_text(
    (
        "UltraFeedback-DPO AlpacaEval-300 "
        "generation complete.\n"
        f"Output SHA256: {output_sha}\n"
        f"Examples: {len(rows)}\n"
        f"Capped: {capped}\n"
        f"Average generated tokens: "
        f"{summary['average_generated_tokens']}\n"
    ),
    encoding="utf-8",
)


print(
    "\n=========================================="
)

print(
    "✓ ULTRAFEEDBACK ALPACAEVAL-300 COMPLETE"
)

print(
    "=========================================="
)

In [ ]:
import json
import random
import hashlib
from pathlib import Path


ROOT = Path(
    "/kaggle/working/small-model-alignment-starter"
)

SFT_PATH = (
    ROOT
    / "outputs"
    / "eval"
    / "sft_qwen2.5_0.5b"
    / "alpaca_eval_300_seed42_sft.json"
)

DPO_PATH = (
    ROOT
    / "outputs"
    / "eval"
    / "ultrafeedback_dpo_seed42"
    / "alpaca_eval_300_seed42_ultrafeedback_dpo.json"
)

PAIR_DIR = (
    ROOT
    / "outputs"
    / "eval"
    / "pairwise"
)

PAIR_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PAIR_PATH = (
    PAIR_DIR
    / "ultrafeedback_dpo_vs_sft_alpaca_eval_300_seed42.jsonl"
)


# ============================================================
# LOAD
# ============================================================

sft_rows = json.loads(
    SFT_PATH.read_text(
        encoding="utf-8"
    )
)

dpo_rows = json.loads(
    DPO_PATH.read_text(
        encoding="utf-8"
    )
)

assert len(sft_rows) == 300
assert len(dpo_rows) == 300


# ============================================================
# ALIGN BY ID
# ============================================================

sft_by_id = {
    row["id"]: row
    for row in sft_rows
}

dpo_by_id = {
    row["id"]: row
    for row in dpo_rows
}


assert set(sft_by_id) == set(dpo_by_id)


# Preserve fixed AlpacaEval order.
ordered_ids = [
    row["id"]
    for row in sft_rows
]


# ============================================================
# RANDOMIZE A/B ORDER DETERMINISTICALLY
# ============================================================

rng = random.Random(42)

pairs = []

a_is_dpo = 0
b_is_dpo = 0


for position, example_id in enumerate(
    ordered_ids
):

    sft = sft_by_id[
        example_id
    ]

    dpo = dpo_by_id[
        example_id
    ]


    # Exact same prompt must be compared.
    assert (
        sft["instruction"]
        == dpo["instruction"]
    )

    assert (
        sft["source_index"]
        == dpo["source_index"]
    )

    assert (
        sft["subset_position"]
        == dpo["subset_position"]
    )


    dpo_first = (
        rng.random() < 0.5
    )


    if dpo_first:

        response_a = (
            dpo["output"]
        )

        response_b = (
            sft["output"]
        )

        model_a = (
            "ultrafeedback_dpo"
        )

        model_b = (
            "sft"
        )

        a_is_dpo += 1

    else:

        response_a = (
            sft["output"]
        )

        response_b = (
            dpo["output"]
        )

        model_a = (
            "sft"
        )

        model_b = (
            "ultrafeedback_dpo"
        )

        b_is_dpo += 1


    pairs.append({
        "pair_id": (
            f"uf_vs_sft_{position:04d}"
        ),

        "example_id": (
            example_id
        ),

        "instruction": (
            sft["instruction"]
        ),

        "response_A": (
            response_a
        ),

        "response_B": (
            response_b
        ),

        # IMPORTANT:
        # This metadata is for translating the judge's
        # A/B decision AFTER judging.
        # We will NEVER include it in the judge prompt.
        "_hidden_metadata": {
            "response_A_model": (
                model_a
            ),

            "response_B_model": (
                model_b
            ),

            "source_index": (
                sft[
                    "source_index"
                ]
            ),

            "subset_position": (
                sft[
                    "subset_position"
                ]
            ),
        },
    })


assert len(pairs) == 300
assert a_is_dpo + b_is_dpo == 300


# ============================================================
# SAVE JSONL
# ============================================================

with open(
    PAIR_PATH,
    "w",
    encoding="utf-8",
) as f:

    for pair in pairs:

        f.write(
            json.dumps(
                pair,
                ensure_ascii=False,
            )
            + "\n"
        )


# ============================================================
# HASH
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


pair_sha = sha256_file(
    PAIR_PATH
)


# ============================================================
# RELOAD + VERIFY
# ============================================================

reloaded = []

with open(
    PAIR_PATH,
    "r",
    encoding="utf-8",
) as f:

    for line in f:

        reloaded.append(
            json.loads(
                line
            )
        )


assert len(reloaded) == 300


print(
    "===== ULTRAFEEDBACK DPO VS SFT PAIRS ====="
)

print(
    "Pairs:",
    len(reloaded)
)

print(
    "DPO is Response A:",
    a_is_dpo
)

print(
    "DPO is Response B:",
    b_is_dpo
)

print(
    "\nSHA256:"
)

print(
    pair_sha
)

print(
    "\nSaved:"
)

print(
    PAIR_PATH
)


print(
    "\n===== FIRST PAIR — JUDGE-VISIBLE FIELDS ====="
)

print(
    json.dumps(
        {
            "pair_id": (
                reloaded[0][
                    "pair_id"
                ]
            ),

            "instruction": (
                reloaded[0][
                    "instruction"
                ][:400]
            ),

            "response_A": (
                reloaded[0][
                    "response_A"
                ][:300]
            ),

            "response_B": (
                reloaded[0][
                    "response_B"
                ][:300]
            ),
        },
        indent=2,
        ensure_ascii=False,
    )
)


print(
    "\n===== FIRST PAIR — HIDDEN MAPPING ====="
)

print(
    reloaded[0][
        "_hidden_metadata"
    ]
)


print(
    "\n✓ Blind pairwise file created"
)

In [1]:
from pathlib import Path
import json
import hashlib
import os
import shutil
import sys

import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM


# ============================================================
# STEP 79
# ULTRAFEEDBACK-DPO -> ALPACAEVAL-300
# ============================================================

markers = list(
    Path("/kaggle/input").rglob(
        "ULTRAFEEDBACK_DPO_COMPLETE.txt"
    )
)

print("Completion markers found:", len(markers))

for p in markers:
    print(p)

assert len(markers) == 1

SAVED_REPO = markers[0].parent

WORKING_REPO = Path(
    "/kaggle/working/small-model-alignment-starter"
)

print("\nSaved repo:", SAVED_REPO)
print("Working repo:", WORKING_REPO)


# ============================================================
# RESTORE
# ============================================================

if WORKING_REPO.exists():
    shutil.rmtree(WORKING_REPO)

print("\nRestoring repository...")

shutil.copytree(
    SAVED_REPO,
    WORKING_REPO,
)

print("✓ Repository restored")


os.chdir(WORKING_REPO)

sys.path.insert(
    0,
    str(WORKING_REPO),
)

from data_utils import format_instruction_prompt


# ============================================================
# PATHS
# ============================================================

UF_BEST = (
    WORKING_REPO
    / "outputs"
    / "ultrafeedback_dpo_seed42"
    / "best_checkpoint"
)

EVAL_PATH = (
    WORKING_REPO
    / "eval_sets"
    / "alpaca_eval_300_seed42.json"
)

OUT_DIR = (
    WORKING_REPO
    / "outputs"
    / "eval"
    / "ultrafeedback_dpo_seed42"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

OUT_PATH = (
    OUT_DIR
    / "alpaca_eval_300_seed42_ultrafeedback_dpo.json"
)


assert UF_BEST.exists()
assert EVAL_PATH.exists()

if OUT_PATH.exists():
    raise RuntimeError(
        f"Output already exists: {OUT_PATH}"
    )


# ============================================================
# DEVICE
# ============================================================

assert torch.cuda.is_available()

DEVICE = torch.device("cuda:0")

print(
    "\nGPU:",
    torch.cuda.get_device_name(0)
)


# ============================================================
# LOAD FIXED EVAL SET
# ============================================================

eval_rows = json.loads(
    EVAL_PATH.read_text(
        encoding="utf-8"
    )
)

assert len(eval_rows) == 300

print(
    "Evaluation prompts:",
    len(eval_rows)
)


# ============================================================
# LOAD MODEL
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    UF_BEST,
    use_fast=True,
)

print(
    "\nLoading UltraFeedback-DPO best checkpoint..."
)

model = AutoModelForCausalLM.from_pretrained(
    UF_BEST,
    dtype=torch.float32,
)

model.to(DEVICE)
model.eval()

print("✓ Model loaded")


# ============================================================
# FROZEN DECODING
# SAME AS BASE / SFT
# ============================================================

GENERATION_CONFIG = {
    "do_sample": False,
    "max_new_tokens": 256,
    "repetition_penalty": 1.0,
    "no_repeat_ngram_size": 4,
}

print("\nGeneration config:")

print(
    json.dumps(
        GENERATION_CONFIG,
        indent=2,
    )
)


# ============================================================
# GENERATE
# ============================================================

outputs = []

with torch.inference_mode():

    for row in tqdm(
        eval_rows,
        desc="UltraFeedback-DPO AlpacaEval",
    ):

        instruction = row["instruction"]

        prompt = format_instruction_prompt(
            instruction
        )

        encoded = tokenizer(
            prompt,
            return_tensors="pt",
            add_special_tokens=False,
        )

        prompt_token_count = (
            encoded["input_ids"].shape[1]
        )

        encoded = {
            key: value.to(DEVICE)
            for key, value in encoded.items()
        }

        generated = model.generate(
            **encoded,
            do_sample=False,
            max_new_tokens=256,
            repetition_penalty=1.0,
            no_repeat_ngram_size=4,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

        new_tokens = generated[
            0,
            prompt_token_count:
        ]

        generated_token_count = int(
            new_tokens.shape[0]
        )

        output_text = tokenizer.decode(
            new_tokens,
            skip_special_tokens=True,
        ).strip()

        hit_max_new_tokens = (
            generated_token_count >= 256
        )

        outputs.append({
            "id": row["id"],
            "instruction": instruction,
            "output": output_text,
            "generator": (
                "qwen2.5-0.5b-ultrafeedback-dpo"
            ),
            "dataset": "alpaca_eval",
            "source_index": row["source_index"],
            "subset_position": row["subset_position"],
            "prompt_token_count": prompt_token_count,
            "generated_token_count": generated_token_count,
            "hit_max_new_tokens": hit_max_new_tokens,
        })


# ============================================================
# SAVE
# ============================================================

OUT_PATH.write_text(
    json.dumps(
        outputs,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("\n✓ Saved:", OUT_PATH)
print("Rows:", len(outputs))

assert len(outputs) == 300

Completion markers found: 1
/kaggle/input/notebooks/muhammadhashirsalman/notebook06adee0aec/small-model-alignment-starter/ULTRAFEEDBACK_DPO_COMPLETE.txt

Saved repo: /kaggle/input/notebooks/muhammadhashirsalman/notebook06adee0aec/small-model-alignment-starter
Working repo: /kaggle/working/small-model-alignment-starter

Restoring repository...
✓ Repository restored

GPU: Tesla T4
Evaluation prompts: 300

Loading UltraFeedback-DPO best checkpoint...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

✓ Model loaded

Generation config:
{
  "do_sample": false,
  "max_new_tokens": 256,
  "repetition_penalty": 1.0,
  "no_repeat_ngram_size": 4
}


UltraFeedback-DPO AlpacaEval:   0%|          | 0/300 [00:00<?, ?it/s]


✓ Saved: /kaggle/working/small-model-alignment-starter/outputs/eval/ultrafeedback_dpo_seed42/alpaca_eval_300_seed42_ultrafeedback_dpo.json
Rows: 300


In [2]:
from pathlib import Path
import json
import hashlib


ROOT = Path(
    "/kaggle/working/small-model-alignment-starter"
)

OUT_PATH = (
    ROOT
    / "outputs"
    / "eval"
    / "ultrafeedback_dpo_seed42"
    / "alpaca_eval_300_seed42_ultrafeedback_dpo.json"
)

MANIFEST = (
    ROOT
    / "PA2_RESUME_STATE.json"
)


rows = json.loads(
    OUT_PATH.read_text(
        encoding="utf-8"
    )
)

assert len(rows) == 300


def repeated_ngram_fraction(text, n=4):

    tokens = text.split()

    if len(tokens) < n:
        return 0.0

    ngrams = [
        tuple(tokens[i:i+n])
        for i in range(
            len(tokens) - n + 1
        )
    ]

    if not ngrams:
        return 0.0

    return (
        len(ngrams) - len(set(ngrams))
    ) / len(ngrams)


generated_lengths = [
    row["generated_token_count"]
    for row in rows
]

capped = sum(
    bool(row["hit_max_new_tokens"])
    for row in rows
)

repeat_fracs = [
    repeated_ngram_fraction(
        row["output"],
        n=4,
    )
    for row in rows
]


summary = {
    "examples": len(rows),
    "hit_max_new_tokens": capped,
    "hit_max_new_tokens_fraction": (
        capped / len(rows)
    ),
    "average_generated_tokens": (
        sum(generated_lengths)
        / len(generated_lengths)
    ),
    "mean_repeated_4gram_fraction": (
        sum(repeat_fracs)
        / len(repeat_fracs)
    ),
    "repeated_4gram_fraction_over_20pct": (
        sum(x > 0.20 for x in repeat_fracs)
    ),
    "repeated_4gram_fraction_over_40pct": (
        sum(x > 0.40 for x in repeat_fracs)
    ),
    "decoding": {
        "do_sample": False,
        "max_new_tokens": 256,
        "repetition_penalty": 1.0,
        "no_repeat_ngram_size": 4,
    },
}


print(
    "===== ULTRAFEEDBACK-DPO ALPACAEVAL SUMMARY ====="
)

print(
    json.dumps(
        summary,
        indent=2,
    )
)


def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


output_sha = sha256_file(
    OUT_PATH
)

print("\nSHA256:")
print(output_sha)


# ============================================================
# UPDATE MANIFEST
# ============================================================

state = json.loads(
    MANIFEST.read_text(
        encoding="utf-8"
    )
)

relative_output = (
    "outputs/eval/"
    "ultrafeedback_dpo_seed42/"
    "alpaca_eval_300_seed42_"
    "ultrafeedback_dpo.json"
)

state["critical_files"][relative_output] = {
    "sha256": output_sha
}

state["current_state"] = (
    "SFT, HH-RLHF DPO, and UltraFeedback DPO "
    "training are complete. UltraFeedback-DPO "
    "generation on the fixed AlpacaEval-300 "
    "evaluation set is also complete using the "
    "same frozen decoding as the Base/SFT runs."
)

state["next_stage"] = (
    "Build the blind SFT-vs-UltraFeedback-DPO "
    "pairwise comparison file, then run the "
    "required Kaggle judge."
)

MANIFEST.write_text(
    json.dumps(
        state,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# COMPLETION MARKER
# ============================================================

marker = (
    ROOT
    / "ULTRAFEEDBACK_ALPACAEVAL_COMPLETE.txt"
)

marker.write_text(
    (
        "UltraFeedback-DPO AlpacaEval-300 "
        "generation complete.\n"
        f"Output SHA256: {output_sha}\n"
        f"Examples: {len(rows)}\n"
        f"Capped: {capped}\n"
        f"Average generated tokens: "
        f"{summary['average_generated_tokens']}\n"
    ),
    encoding="utf-8",
)

print(
    "\n=========================================="
)

print(
    "✓ ULTRAFEEDBACK ALPACAEVAL-300 COMPLETE"
)

print(
    "=========================================="
)

===== ULTRAFEEDBACK-DPO ALPACAEVAL SUMMARY =====
{
  "examples": 300,
  "hit_max_new_tokens": 155,
  "hit_max_new_tokens_fraction": 0.5166666666666667,
  "average_generated_tokens": 193.39666666666668,
  "mean_repeated_4gram_fraction": 0.0011524947641034908,
  "repeated_4gram_fraction_over_20pct": 0,
  "repeated_4gram_fraction_over_40pct": 0,
  "decoding": {
    "do_sample": false,
    "max_new_tokens": 256,
    "repetition_penalty": 1.0,
    "no_repeat_ngram_size": 4
  }
}

SHA256:
e89ba1671b8c683a1b78c53d1bc980b4dfd26608e54609f1420bd8880cdf7e95

✓ ULTRAFEEDBACK ALPACAEVAL-300 COMPLETE


In [3]:
import json
import random
import hashlib
from pathlib import Path


ROOT = Path(
    "/kaggle/working/small-model-alignment-starter"
)

SFT_PATH = (
    ROOT
    / "outputs"
    / "eval"
    / "sft_qwen2.5_0.5b"
    / "alpaca_eval_300_seed42_sft.json"
)

DPO_PATH = (
    ROOT
    / "outputs"
    / "eval"
    / "ultrafeedback_dpo_seed42"
    / "alpaca_eval_300_seed42_ultrafeedback_dpo.json"
)

PAIR_DIR = (
    ROOT
    / "outputs"
    / "eval"
    / "pairwise"
)

PAIR_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PAIR_PATH = (
    PAIR_DIR
    / "ultrafeedback_dpo_vs_sft_alpaca_eval_300_seed42.jsonl"
)


# ============================================================
# LOAD
# ============================================================

sft_rows = json.loads(
    SFT_PATH.read_text(
        encoding="utf-8"
    )
)

dpo_rows = json.loads(
    DPO_PATH.read_text(
        encoding="utf-8"
    )
)

assert len(sft_rows) == 300
assert len(dpo_rows) == 300


# ============================================================
# ALIGN BY ID
# ============================================================

sft_by_id = {
    row["id"]: row
    for row in sft_rows
}

dpo_by_id = {
    row["id"]: row
    for row in dpo_rows
}

assert set(sft_by_id) == set(dpo_by_id)

ordered_ids = [
    row["id"]
    for row in sft_rows
]


# ============================================================
# DETERMINISTIC RANDOM A/B ORDER
# ============================================================

rng = random.Random(42)

pairs = []

a_is_dpo = 0
b_is_dpo = 0


for position, example_id in enumerate(
    ordered_ids
):

    sft = sft_by_id[example_id]
    dpo = dpo_by_id[example_id]

    assert (
        sft["instruction"]
        == dpo["instruction"]
    )

    assert (
        sft["source_index"]
        == dpo["source_index"]
    )

    assert (
        sft["subset_position"]
        == dpo["subset_position"]
    )


    dpo_first = (
        rng.random() < 0.5
    )


    if dpo_first:

        response_a = dpo["output"]
        response_b = sft["output"]

        model_a = "ultrafeedback_dpo"
        model_b = "sft"

        a_is_dpo += 1

    else:

        response_a = sft["output"]
        response_b = dpo["output"]

        model_a = "sft"
        model_b = "ultrafeedback_dpo"

        b_is_dpo += 1


    pairs.append({
        "pair_id": (
            f"uf_vs_sft_{position:04d}"
        ),

        "example_id": example_id,

        "instruction": (
            sft["instruction"]
        ),

        "response_A": response_a,

        "response_B": response_b,

        "_hidden_metadata": {
            "response_A_model": model_a,
            "response_B_model": model_b,
            "source_index": (
                sft["source_index"]
            ),
            "subset_position": (
                sft["subset_position"]
            ),
        },
    })


assert len(pairs) == 300
assert a_is_dpo + b_is_dpo == 300


# ============================================================
# SAVE JSONL
# ============================================================

with open(
    PAIR_PATH,
    "w",
    encoding="utf-8",
) as f:

    for pair in pairs:

        f.write(
            json.dumps(
                pair,
                ensure_ascii=False,
            )
            + "\n"
        )


# ============================================================
# HASH
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


pair_sha = sha256_file(
    PAIR_PATH
)


# ============================================================
# VERIFY
# ============================================================

with open(
    PAIR_PATH,
    "r",
    encoding="utf-8",
) as f:

    reloaded = [
        json.loads(line)
        for line in f
        if line.strip()
    ]


assert len(reloaded) == 300


print(
    "===== ULTRAFEEDBACK DPO VS SFT PAIRS ====="
)

print("Pairs:", len(reloaded))
print("DPO is Response A:", a_is_dpo)
print("DPO is Response B:", b_is_dpo)

print("\nSHA256:")
print(pair_sha)

print("\nSaved:")
print(PAIR_PATH)

print(
    "\n===== FIRST PAIR — HIDDEN MAPPING ====="
)

print(
    reloaded[0][
        "_hidden_metadata"
    ]
)

print(
    "\n✓ Blind pairwise file created"
)

===== ULTRAFEEDBACK DPO VS SFT PAIRS =====
Pairs: 300
DPO is Response A: 151
DPO is Response B: 149

SHA256:
3312bc56fa19f06d64853ffa88519009771c1855d92a46f0a560775e7f66ffa9

Saved:
/kaggle/working/small-model-alignment-starter/outputs/eval/pairwise/ultrafeedback_dpo_vs_sft_alpaca_eval_300_seed42.jsonl

===== FIRST PAIR — HIDDEN MAPPING =====
{'response_A_model': 'sft', 'response_B_model': 'ultrafeedback_dpo', 'source_index': 654, 'subset_position': 0}

✓ Blind pairwise file created
